# End-to-End ETL Workflow

This notebook demonstrates a complete Extract-Transform-Load pipeline using the hybrid Rust+Python DataFrame engine.

## Scenario

We're processing e-commerce order data:
- **Extract**: Load raw orders from CSV
- **Transform**: Clean data, calculate metrics, filter
- **Load**: Aggregate and export to Parquet

In [ ]:
# Setup
from hyperframe import DataFrame, read_csv
import time

print("SDK loaded - Rust engine ready!")

## Step 1: Extract

In [ ]:
# Load raw order data
start = time.time()
orders = read_csv("data/raw/orders.csv")
load_time = time.time() - start

print(f"Loaded {orders.shape[0]:,} rows in {load_time:.2f}s")
print(f"Columns: {orders.columns}")

In [ ]:
# Preview the raw data
orders.head()

In [ ]:
# Check data quality
orders.info()

## Step 2: Transform

In [ ]:
# Handle missing values
print("Null counts before cleaning:")
for col in orders.columns:
    null_count = orders[col].isnull().sum()
    if null_count > 0:
        print(f"  {col}: {null_count:,}")

# Fill nulls with sensible defaults
orders.fill_na("quantity", 1)
orders.fill_na("unit_price", 0.0)

In [ ]:
# Add derived columns
# Vectorized arithmetic in Rust
orders["total_value"] = orders["quantity"] * orders["unit_price"]
orders["order_month"] = orders["order_date"].str.slice(0, 7)  # "YYYY-MM"

print("Added columns: total_value, order_month")
orders[["order_id", "quantity", "unit_price", "total_value", "order_month"]].head()

In [ ]:
# Filter to meaningful orders (value > $100)
print(f"Before filter: {orders.shape[0]:,} rows")
orders = orders[orders["total_value"] > 100]
print(f"After filter: {orders.shape[0]:,} rows")

## Step 3: Load - Aggregations

In [ ]:
# Regional summary
regional = orders.group_by("region").agg({
    "total_value": ["sum", "mean", "count"],
    "quantity": "sum"
})

print("=== Regional Summary ===")
regional

In [ ]:
# Monthly trends
monthly = orders.group_by("order_month").agg({
    "total_value": "sum",
    "order_id": "count"
}).sort_by("order_month")

print("=== Monthly Trends ===")
monthly.head(12)

In [ ]:
# Product category analysis
if "category" in orders.columns:
    category_stats = orders.group_by("category").agg({
        "total_value": ["sum", "mean"],
        "order_id": "count"
    }).sort_by(("total_value", "sum"), ascending=False)
    
    print("=== Category Performance ===")
    category_stats

## Step 4: Export

In [ ]:
# Write processed data to Parquet
orders.to_parquet(
    "data/processed/orders.parquet",
    compression="snappy",
    row_group_size=100_000
)
print("Wrote orders to data/processed/orders.parquet")

In [ ]:
# Write summary to CSV
regional.to_pandas().to_csv("data/processed/regional_summary.csv", index=False)
monthly.to_pandas().to_csv("data/processed/monthly_trends.csv", index=False)

print("Wrote summary files to data/processed/")

## Performance Summary

In [ ]:
# Estimate memory usage (columnar layout, ~8 bytes per fixed-width value)
n_rows, n_cols = orders.shape
estimated_mb = (n_rows * n_cols * 8) / (1024 * 1024)
print(f"Memory estimate: ~{estimated_mb:.0f} MB for {n_rows:,} rows × {n_cols} cols")

In [ ]:
# Pipeline timing summary
print("""
=== PIPELINE COMPLETE ===

Input:  data/raw/orders.csv
Output: data/processed/orders.parquet

Transformations applied:
  - Null filling (quantity, unit_price)
  - Derived columns (total_value, order_month)
  - Filter (total_value > 100)

Aggregations computed:
  - Regional summary
  - Monthly trends
  - Category performance
""")